# ColorSAR Model 1 — training on Kaggle (2x Tesla T4)

This notebook is a thin driver: all model, data, split and training code lives in the
repository (`train_colorizer.py` and friends), so the notebook can never drift from the
code used for evaluation and inference.

Setup: enable **Internet** (to clone the repo) *or* attach the repo as a Kaggle dataset
and set `REPO_DIR`; attach `requiemonk/sentinel12-image-pairs-segregated-by-terrain`.


In [ ]:
# 1. GPUs + dataset location
import os, subprocess, sys, torch
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

# dataset root = the folder whose children are the terrain folders (each holding s1/ and s2/)
dataset_root = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "s1" in dirs and "s2" in dirs:
        dataset_root = os.path.dirname(root)
        break
assert dataset_root, "dataset not found under /kaggle/input"
print("dataset root:", dataset_root)


In [ ]:
# 2. Repository (cloned, or attached as a dataset)
REPO_URL = "https://github.com/Priyanshaur/ColorSAR.git"
REPO_DIR = "/kaggle/working/ColorSAR"     # or e.g. "/kaggle/input/colorsar-repo"
if not os.path.exists(os.path.join(REPO_DIR, "train_colorizer.py")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image", "opencv-python-headless"])
os.chdir(REPO_DIR)
print("working in", os.getcwd())


In [ ]:
# 3. Train Model 1 with the published recipe (defaults of Train_model1.py:
#    all canonical train pairs, 20 epochs, batch 64, Adam 5e-4 + cosine).
#    DataParallel across both T4s is automatic. Add --train_adapters / --imagenet_norm
#    to try the improved variants (recorded in the checkpoint, applied at inference).
OUT = "/kaggle/working/model1_run"
!python Train_model1.py --root "{dataset_root}" --out "{OUT}" --num_workers 4


In [ ]:
# 4. Results
from IPython.display import Image, display
for f in ["loss_curve.png", "sample_predictions.png"]:
    display(Image(os.path.join(OUT, f)))
# Download model_1.pth, split.json, history.json and the PNGs from /kaggle/working/model1_run,
# put them in model1_run/ locally, then:  python evaluate.py --model 1 --per_terrain
